In [1]:
import json
from collections import Counter

with open("meged3.json", "r", encoding="utf-8") as f:
    data = json.load(f)

counter = Counter(item["tv_form"] for item in data if item.get("tv_form"))

VALID_CLASSES = [tv for tv, count in counter.items() if count >= 20]

filtered_data = [
    item for item in data
    if item.get("tv_form") in VALID_CLASSES
]

print("Total after filtering:", len(filtered_data))
print("Remaining classes:", len(set(x["tv_form"] for x in filtered_data)))

Total after filtering: 3209
Remaining classes: 16


In [2]:
from sklearn.preprocessing import LabelEncoder

texts = [x["text"] for x in data if x.get("sem_type")]
labels = [x["sem_type"] for x in data if x.get("sem_type")]

label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(labels)

num_labels = len(label_encoder.classes_)
print("Number of classes:", num_labels)
print("Classes:", list(label_encoder.classes_))

id2label = {i: label for i, label in enumerate(label_encoder.classes_)}
label2id = {label: i for i, label in enumerate(label_encoder.classes_)}

Number of classes: 4
Classes: [np.str_('DEPENDENCE'), np.str_('EMOTIVE_REACTION'), np.str_('JUSTIFICATION'), np.str_('SANCTION')]


In [3]:
import json
from collections import Counter

with open("meged3.json", "r", encoding="utf-8") as f:
    data = json.load(f)

sem_labels = [item["sem_type"] for item in data if item.get("sem_type")]

counter = Counter(sem_labels)

total = sum(counter.values())

print("=" * 50)
print("TOTAL SAMPLES:", total)
print("UNIQUE SEM_TYPES:", len(counter))
print("=" * 50)

print("\nDISTRIBUTION:\n")

for label, count in counter.most_common():
    percent = round(count / total * 100, 2)
    print(f"{label:15}  Count: {count:<5} | {percent}%")

TOTAL SAMPLES: 3223
UNIQUE SEM_TYPES: 4

DISTRIBUTION:

DEPENDENCE       Count: 2234  | 69.31%
EMOTIVE_REACTION  Count: 598   | 18.55%
SANCTION         Count: 198   | 6.14%
JUSTIFICATION    Count: 193   | 5.99%


In [4]:
from sklearn.model_selection import train_test_split
from datasets import Dataset

train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts,
    encoded_labels,
    test_size=0.2,
    random_state=42,
    stratify=encoded_labels
)

train_dataset = Dataset.from_dict({
    "text": train_texts,
    "labels": train_labels
})

test_dataset = Dataset.from_dict({
    "text": test_texts,
    "labels": test_labels
})

c:\Users\egisb\voice1\kazFineTune\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [6]:
from transformers import AutoTokenizer

model_ckpt = "Eraly-ml/KazBERT"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map: 100%|██████████| 645/645 [00:00<00:00, 15440.21 examples/s]


In [7]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_ckpt,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at Eraly-ml/KazBERT and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
import torch
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

def compute_metrics(p):
    logits = torch.tensor(p.predictions)

    probs = torch.nn.functional.softmax(logits, dim=1).numpy()

    preds = np.argmax(probs, axis=1)
    labels = p.label_ids

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "roc_auc": roc_auc_score(labels, probs, multi_class="ovr", average="macro")
    }

In [13]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./kazbert_sem_train_final",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=8,
    weight_decay=0.01,
    fp16=True,
    load_best_model_at_end=True,
    logging_dir="./logs"
)

In [14]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

C:\Users\egisb\AppData\Local\Temp\ipykernel_9060\4223154140.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Roc Auc
1,No log,0.014416,0.998450,0.998683,0.999304
2,No log,0.028167,0.996899,0.995239,0.999331
3,No log,0.024854,0.996899,0.997357,0.999357
4,0.010100,0.030207,0.995349,0.993914,0.999337
5,0.010100,0.046186,0.995349,0.993914,0.997556
6,0.010100,0.036553,0.995349,0.993914,0.999413
7,0.003200,0.033940,0.995349,0.993914,0.999396
8,0.003200,0.034613,0.995349,0.993914,0.999349


TrainOutput(global_step=1296, training_loss=0.0051503053774344335, metrics={'train_runtime': 230.5488, 'train_samples_per_second': 89.456, 'train_steps_per_second': 5.621, 'total_flos': 1356624962174976.0, 'train_loss': 0.0051503053774344335, 'epoch': 8.0})

In [15]:
results = trainer.evaluate()
print(results)

{'eval_loss': 0.014416423626244068, 'eval_accuracy': 0.9984496124031008, 'eval_macro_f1': 0.9986827003944929, 'eval_roc_auc': 0.9993039656714153, 'eval_runtime': 1.7475, 'eval_samples_per_second': 369.099, 'eval_steps_per_second': 23.462, 'epoch': 8.0}


In [16]:
from sklearn.metrics import classification_report

predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)

print(classification_report(
    test_labels,
    preds,
    target_names=label_encoder.classes_
))

                  precision    recall  f1-score   support

      DEPENDENCE       1.00      1.00      1.00       447
EMOTIVE_REACTION       0.99      1.00      1.00       120
   JUSTIFICATION       1.00      1.00      1.00        38
        SANCTION       1.00      1.00      1.00        40

        accuracy                           1.00       645
       macro avg       1.00      1.00      1.00       645
    weighted avg       1.00      1.00      1.00       645



In [17]:
trainer.save_model("kazbert_sem_model_final")
tokenizer.save_pretrained("kazbert_sem_model_final")

('kazbert_sem_model_final\\tokenizer_config.json',
 'kazbert_sem_model_final\\special_tokens_map.json',
 'kazbert_sem_model_final\\vocab.txt',
 'kazbert_sem_model_final\\added_tokens.json',
 'kazbert_sem_model_final\\tokenizer.json')

In [18]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

model_dir = "kazbert_sem_model_final"

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)

tv_pipeline = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer
)

Device set to use cuda:0


In [ ]:
sentence = "Ғылыми гипотезаның толық дәлелденбейтініне қынжылдық,  зерттеу әдіснамасы жетілдіруді талап етті."
prediction = tv_pipeline(sentence)

print(prediction)

[{'label': 'DEPENDENCE', 'score': 0.9998899698257446}]


In [23]:
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification

sem_tokenizer = AutoTokenizer.from_pretrained("kazbert_sem_model_final")
sem_model = AutoModelForSequenceClassification.from_pretrained("kazbert_sem_model_final")

# Without top_k=None
pipe_default = pipeline("text-classification", model=sem_model, tokenizer=sem_tokenizer)
# With top_k=None  
pipe_topk = pipeline("text-classification", model=sem_model, tokenizer=sem_tokenizer, top_k=None)

sentence = "Қысқа да әрі нақты, көпке түсінікті болғандықтан студенттерге бір деммен оқу қиынға соқпауы мүмкін."

print("default:", pipe_default(sentence))
print("top_k=None:", pipe_topk(sentence))
print("top_k=None [0]:", pipe_topk(sentence)[0])

Device set to use cuda:0
Device set to use cuda:0


default: [{'label': 'DEPENDENCE', 'score': 0.9998899698257446}]
top_k=None: [[{'label': 'DEPENDENCE', 'score': 0.9998899698257446}, {'label': 'JUSTIFICATION', 'score': 4.3395284592406824e-05}, {'label': 'EMOTIVE_REACTION', 'score': 3.578501855372451e-05}, {'label': 'SANCTION', 'score': 3.086768265347928e-05}]]
top_k=None [0]: [{'label': 'DEPENDENCE', 'score': 0.9998899698257446}, {'label': 'JUSTIFICATION', 'score': 4.3395284592406824e-05}, {'label': 'EMOTIVE_REACTION', 'score': 3.578501855372451e-05}, {'label': 'SANCTION', 'score': 3.086768265347928e-05}]
